# Rust 114: Smart Pointers and Interior Mutability

Chapter 14 of the Rust track, and the chapter about *sharing* — the two powers the borrow checker has been refusing on your behalf since chapter 05: many owners of one value, and mutation through a shared handle. **Smart pointers** (`Box`, `Rc`, `Arc`) are structs that own a heap allocation and control what happens when the last handle drops; **interior mutability** (`Cell`, `RefCell`) moves the borrow check from compile time to run time *inside* an otherwise-immutable shell. Chapter 12's laziness counter smuggled one across the border (`Rc<Cell<usize>>`); this chapter owns the full tour — and chapter 13's fence gets its complement: smart pointers share data *across* boundaries without publishing internals. Everything here is std-only and in-process.

Everything here is a **runnable example**. Read the markdown, run the cell below it, and change nothing — the practice twin of this notebook is `14_smart-pointers-and-interior-mutability_p.ipynb`.

## Who this is written for

You are assumed to be a competent programmer arriving from Python and C++. Python references everything and frees nothing explicitly; C++ hands you `unique_ptr`/`shared_ptr` with copy semantics that blur the ownership story. What is new in Rust: a *plain* reference `&T` is a borrow, not an owner — the smart pointers are the only way a value can outlive the function that made it, and each pointer type is a distinct *contract*: `Box` = sole owner, heap-resident; `Rc` = many readers, one allocation, no mutation; `RefCell` = the borrow rules enforced per-access instead of per-compile. None of them are garbage collectors — the drop timing is still exactly determined, which is why the cycle in §10 leaks and why the fix is a *type change*, not a cleanup pass.

## How to read this notebook

- Markdown cells explain; code cells demonstrate. **Run each code cell as you reach it.** Cells are independent — no sandbox, no files touched, so restart-and-run-from-the-top always works.
- Code that intentionally *fails* is shown as a comment, together with the diagnostic you would get. §11 is the catalogue.
- The running example is a `Meter` family — an owner that counts — introduced in §1 and reused through §10's cycle.
- One honesty note runs through the chapter: `Rc`, `Cell`, `RefCell` and `Weak` live in single-threaded land. `Arc` appears once (§9) mostly to show where the thread boundary cuts; the threading tour itself is chapter 16.

## The one idea to internalise

**Ownership is a compile-time fact; smart pointers move pieces of it to runtime — deliberately, and with a receipt.** `Box` moves the *storage* to the heap but keeps the single-owner rule. `Rc` relaxes the *one-owner* rule and pays with an immutable, counted shell. `RefCell` relaxes the *compile-time* borrow check and pays with runtime panics. Stack them — `Rc<RefCell<T>>` — and you can share *and* mutate, at the price of reasoning about borrows by hand. Every relaxation is still checked somewhere: `Rc` panics on underflow only if you `try_unwrap` wrong, `RefCell` panics the moment two conflicting borrows overlap, and a reference cycle leaks because drop is *reference-driven*, not garbage-collected.


### Contents

- [0. Notebook Conventions: How These Cells Run](#0-notebook-conventions-how-these-cells-run)
- [1. `Box`: The Heap Pointer](#1-box-the-heap-pointer)
- [2. `Deref` and `DerefMut`: The Coercion Machine](#2-deref-and-derefmut-the-coercion-machine)
- [3. `Drop`: The Destructor](#3-drop-the-destructor)
- [4. `Rc`: Many Owners, One Allocation](#4-rc-many-owners-one-allocation)
- [5. `Cell`: Copy Into and Copy Out](#5-cell-copy-into-and-copy-out)
- [6. `RefCell`: The Runtime Borrow Check](#6-refcell-the-runtime-borrow-check)
- [7. `Rc<RefCell<T>>`: Sharing and Mutating](#7-rcrefcellt-sharing-and-mutating)
- [8. `OnceCell` and `OnceLock`: Write Once](#8-oncecell-and-oncelock-write-once)
- [9. `Arc` and the Thread Boundary](#9-arc-and-the-thread-boundary)
- [10. Cycles and `Weak`: The Leak](#10-cycles-and-weak-the-leak)
- [11. Reading the Compiler: E-Codes](#11-reading-the-compiler-e-codes)
- [12. Mental Model: The Receipts](#12-mental-model-the-receipts)
- [13. Cheat Sheet](#13-cheat-sheet)
- [14. Review](#14-review)

If you are returning to this chapter later, §13 is the one-page summary, §11 the diagnostics catalogue, and §7 the stacked-pointer judgment call.


## 0. Notebook Conventions: How These Cells Run

This notebook runs on the **evcxr** kernel, which wraps each cell into a generated program and compiles it incrementally. Three consequences matter for every cell below:

- **There is no `fn main`.** The kernel supplies one; you are typing its body.
- **State persists across cells.** A `let` from cell 4 is still live in cell 20 — including `use` statements.
- **Items are order-independent, statements are not.** Two `fn`s may call each other regardless of order; two `let`s may not use each other's names before they run.

There is **one namespace per notebook**. Two items with the same name collide exactly as they would in one source file, even in different sections. Every struct, module and function name below is unique across the notebook — a re-declaration is the same redefinition error as a re-`fn` (E0428). If you re-run a cell that defines an item, it is redefined; on a redefinition error, restart the kernel and run from the top.

A failing cell does not poison the kernel — the deliberate compile errors in this chapter are comments, so every cell still runs.

### Drop and the kernel: one honest caveat

`Drop` timing is *deterministic* — end of scope, or `drop(value)` — and §3 proves it. The kernel's generated wrapper is one big scope per cell, with one exception worth knowing before it surprises you: evcxr does not run destructors of variables left live at the *end* of a cell, because the wrapper keeps them alive for the next cell (state persists, §0's second rule). `mem::drop` and explicit inner scopes show the timing honestly; a value dropped "when the cell ends" may report nothing until the kernel discards it.


## 1. `Box`: The Heap Pointer

`Box<T>` is the smallest smart pointer: one owned `T` on the heap, one pointer-sized handle on the stack, and **zero runtime machinery** — no counts, no locks, no weak table. `Box::new(v)` moves `v` onto the heap; the `Box` owns it; dropping the `Box` drops the `T` and frees the allocation. That is the entire contract, and it buys three things: a *fixed-size handle* for data whose size the compiler cannot know statically (the recursive `List` below — without the `Box`, the variant would contain itself, E0072), *ownership you can move around* cheaply (the handle moves, the payload stays), and a place for large values so moves stay pointer-sized.

| Expression | Size (probed) | The reading |
|---|---|---|
| `Box<i32>` | 8 | one pointer, regardless of `T` |
| `Box<[u8; 100]>` | 8 | the 100 bytes are on the heap; the handle never grows |
| `Box<dyn Display>` | 16 | fat pointer: data + vtable (chapter 10's dyn costs) |
| `&i32` | 8 | a borrow is pointer-sized too — ownership is the difference |

### The recursive type, and the move-out


In [ ]:
// 1.1 Box: the recursive type Box makes legal, and the value Box lets you move out.
#[derive(Debug)]
enum Menu {
    Course(i32, Box<Menu>), // without Box, Course would contain itself: E0072
    Done,
}
let dinner = Menu::Course(1, Box::new(Menu::Course(2, Box::new(Menu::Done))));
// the walk the Box makes possible: recursion through the heap tail
fn menu_sum(menu: &Menu) -> i32 {
    match menu {
        Menu::Course(n, next) => n + menu_sum(next),
        Menu::Done => 0,
    }
}
assert_eq!(menu_sum(&dinner), 3); // both fields read, all the way down
// Box is a real owner: you may move the payload OUT of it (a &T never allows that)
let boxed = Box::new("ticket".to_string());
let unboxed: String = *boxed; // deref-move: the String leaves the heap allocation
assert_eq!(unboxed, "ticket");
// and the size table's first row, verified:
assert_eq!(std::mem::size_of::<Box<[u8; 100]>>(), 8);
println!("Box: recursive type built, payload moved out, handle still 8 bytes");


**`E0072` is the chapter's first gate**: a variant that holds itself has infinite size, and the diagnostic proposes exactly this fix — "insert some indirection (e.g., a `Box`, `Rc`, or `&`) to make `Menu` representable". The **move-out** line is the other half of the lesson: `*boxed` consumes the `Box` and hands you the payload — ownership is *transferred*, not copied, which is why the same move through a plain `&String` would be E0507 (chapter 12's `cannot move out of` catalogue entry). `Box` is the only smart pointer with this deref-move privilege; `Rc` refuses it by construction (§4), because moving out would orphan the other owners.

## 2. `Deref` and `DerefMut`: The Coercion Machine

`Deref` is the trait behind the `*`: `fn deref(&self) -> &Self::Target`. Every smart pointer implements it (`Box<T>` targets `T`, `Rc<T>` targets `T`, `String` targets `str`), and the compiler inserts deref calls **automatically** at two sites — the *deref coercions*: passing `&Box<String>` where `&String` is expected, and `&String` where `&str` is expected, is one inserted `.deref()` apart. Method calls coerce the same way: `boxed.len()` resolves through `String`'s methods without a single `&*`. The coercion is the ergonomic miracle and the mental-model trap: the code reads as if `Rc<String>` *were* a `String`, but it is still a handle you can only read through — `clone` is the explicit move.

Forgetting `Target` is not the error you'd guess: the compiler reports **E0046** (`not all trait items implemented, missing: Target`) rather than an E0106 lifetime error — the associated type is an *item*, not a lifetime annotation.

### The ladder, exercised


In [ ]:
// 2.1 Deref: a hand-rolled smart pointer, then the coercions it unlocks.
use std::ops::{Deref, DerefMut};

struct MeterBox {
    payload: String, // the "heap" this toy pointer manages
}
impl MeterBox {
    fn new(s: &str) -> Self {
        MeterBox { payload: s.to_string() }
    }
    fn measured(&self) -> usize {
        self.payload.len()
    }
}
impl Deref for MeterBox {
    type Target = String; // the associated type the pointer dereferences to
    fn deref(&self) -> &String {
        &self.payload
    }
}
impl DerefMut for MeterBox {
    fn deref_mut(&mut self) -> &mut String {
        &mut self.payload
    }
}
let mut m = MeterBox::new("wired");
// coercion 1: &MeterBox -> &String -> &str, with no &* in sight
fn takes_str(s: &str) -> usize {
    s.chars().count()
}
assert_eq!(takes_str(&m), 5);
// coercion 2: method resolution walks the Deref chain
m.push_str("-live"); // DerefMut: &mut MeterBox -> &mut String
assert_eq!(m.measured(), 10);
assert_eq!(m.to_uppercase(), "WIRED-LIVE"); // String's own method, via deref
println!("MeterBox derefs: {} — one trait, every method of String", m.payload);


Three reads on that cell. **`type Target` is the whole trait**: the impl says "a `&MeterBox` can also be read as a `&String`", and every coercion below is the compiler exploiting that sentence. **`DerefMut` is the write half** — `m.push_str(..)` needed `&mut String`, and the coercion chain ran through `deref_mut` without ceremony (the compiler borrows `m` mutably and derefs it, exactly as §6's `RefCell` will refuse to do at runtime). **The chain is multi-hop**: `&MeterBox` → `&String` → `&str` happened in one argument position; the ladder has as many rungs as your `Target`s, and `&Rc<String>` reaches `&str` the same way (§4's cell uses it in anger).

## 3. `Drop`: The Destructor

`Drop` is the trait that gives a type a last word: `fn drop(&mut self)`, called **exactly once**, when the value's owner goes away — end of scope, end of the binding, or an explicit `std::drop(value)`. The destructor is not directly callable as a method — `value.drop()` is refused outright (only the runtime drops); `std::mem::drop(value)` *consumes* the value and runs the destructor now, which is the sanctioned early exit. The order is the reverse of declaration: locals drop **last-in, first-out**, so a value declared after its dependency drops first — the stack unwinds like a perfectly disciplined pop. `Rc`'s entire safety story stands on this hook: the count decrements in `drop`, the allocation frees when it reaches zero (§4), and the leak in §10 is what happens when "reaches zero" never comes.

One rule and two refusals: a type may implement `Drop` **only for its own local types** (`impl Drop for i32` is E0117 *and* E0120 — "the `Drop` trait may only be implemented for local structs, enums, and unions"), and only **once** per type (a second impl is E0119, chapter 10's orphan-lesson conflict again). `Drop::drop` also receives `&mut self` and takes no arguments — a `fn drop(&self) {}` signature is E0308 ("method `drop` has an incompatible type for trait").

### The timing, observed


In [ ]:
// 3.1 Drop: LIFO order, early drop via mem::drop, and the Rc tie-in.
use std::mem::drop;

struct Gauge {
    label: &'static str,
}
impl Gauge {
    fn new(label: &'static str) -> Self {
        Gauge { label }
    }
}
impl Drop for Gauge {
    fn drop(&mut self) {
        println!("[drop] {}", self.label);
    }
}
{
    let outer = Gauge::new("outer");
    let inner = Gauge::new("inner");
    assert_eq!(outer.label, "outer"); // the fields stay readable until the drop
    assert_eq!(inner.label, "inner");
    drop(inner); // early: the destructor runs NOW, not at scope end
    println!("(after mem::drop(inner))");
    let late = Gauge::new("late");
    assert_eq!(late.label, "late");
} // the brace: `late` then `outer` drop - last in, first out
println!("(scope ended: the LIFO drops printed above)");


The probed facts, stated plainly: `mem::drop(inner)` printed `[drop] inner` **before** the following `println!` — the destructor is immediate, not queued; locals drop in reverse declaration order (`inner-again`, then `outer`, whenever the kernel discards them — see §0's caveat); and a **fresh inner scope is the honest way to see LIFO in one cell**: bind `a`, `b`, `c` inside `{ .. }` and the drops print `c`, `b`, `a` at the closing brace. `Drop`'s signature refusals live in §11's catalogue (E0117+E0120 for foreign types, E0119 for the second impl). What the destructor is *for* is §4's business: `Rc::drop` is the count decrement, and every smart pointer in this chapter is just a struct with a well-behaved `Drop`.


## 4. `Rc`: Many Owners, One Allocation

(§3's `Gauge` type is reused below as an observable payload: its `[drop]` line is the proof the count reached zero.)

`Rc<T>` — reference counted — is the answer to a question the borrow checker has been refusing all along: *can several owners share one value?* `Rc::new(v)` heap-allocates `v` beside two counters (strong = live owners, weak = §10's watchers). `Rc::clone(&r)` — or `r.clone()`, the same call — copies the *handle* and increments the strong count; it does **not** deep-copy the data. Each handle's `Drop` decrements the count; the payload frees when it hits zero. Probed on this toolchain: new → strong 1; clone → 2, `Rc::ptr_eq(&a, &b)` true (same allocation, not equal values); drop one → 1; `Rc::try_unwrap` while shared → `Err` (the refusal that keeps the count honest); `weak_count` of a fresh `Rc` → 0. Size: 8 bytes — the same pointer width as `Box`, plus the count header living on the *heap* beside the data.

The price of shared ownership is **read-only sharing**: `Rc<T>` hands out only shared borrows — there is no `DerefMut`, no safe way to reach `&mut T`, because two owners could mutate through two handles with no borrow checker between them (§6 relaxes *that*, inside the shell). `Rc` also refuses to cross threads — `Rc<i32>` is neither `Send` nor `Sync` (§9's `Arc` is the counted pointer that does).

### The counts, observed live


In [ ]:
// 4.1 Rc: clone counts up, drop counts down, get_mut/try_unwrap refuse while shared.
use std::rc::Rc;
let base = Rc::new("line".to_string());
assert_eq!(Rc::strong_count(&base), 1); // one owner
let reader = Rc::clone(&base); // increments; no deep copy
assert_eq!(Rc::strong_count(&base), 2);
assert!(Rc::ptr_eq(&base, &reader)); // SAME allocation
assert_eq!((*base).len(), 4); // deref to the payload
assert_eq!(base.len(), 4); // the same deref, coerced through the method call
// the mut refusals, both ways:
let mut alone = Rc::clone(&base);
assert!(Rc::get_mut(&mut alone).is_none()); // two owners -> no &mut
assert!(Rc::try_unwrap(alone).is_err()); // unwrapping a shared Rc fails
// counts fall as handles drop - and the payload frees AT ZERO (§3's Gauge
// payload prints its [drop] the moment the last handle goes):
let tracked = Rc::new(Gauge::new("rc-payload"));
let h2 = Rc::clone(&tracked);
assert_eq!(Rc::strong_count(&tracked), 2);
drop(h2);
assert_eq!(Rc::strong_count(&tracked), 1);
drop(tracked); // count hits zero: the destructor runs HERE
println!("Rc: count fell to zero and [drop] rc-payload printed above");


## 5. `Cell`: Copy Into and Copy Out

`Cell<T>` is interior mutability at its bluntest: `.get()` copies the value out, `.set(v)` copies a new one in — **through a shared reference**. No borrows are handed out at all, which is why `Cell` needs `T: Copy` (`get` would otherwise move out of shared memory) and why it can never panic: there is no borrow state to violate. Chapter 12's laziness counter was exactly this: an `Rc<Cell<usize>>` whose closure called `counter.set(counter.get() + 1)` — shared, mutable, borrowing-closure-friendly, and boring in the best way. Size, probed: `Cell<i32>` is 4 bytes — the counter costs **nothing** over the raw `i32`; `RefCell<i32>` is 16 (the borrow flag rides along, §6).

The two other moves round out the toolbox: `take()` swaps in `T::default()` and hands you the old value, and `get_mut()` — on a `&mut Cell` — is the plain-borrow escape hatch. What `Cell` cannot do is lend: any type that is not `Copy` (`String`, `Vec`, a closure) must go through `RefCell` instead.

### The counter, retired from ch12, explained


In [ ]:
// 5.1 Cell: get/set through a shared reference - the ch12 counter, unmasked.
use std::cell::Cell;
let counter = Cell::new(0usize);
let increment = |by: usize| {
    counter.set(counter.get() + by) // shared borrow, mutating call: Cell only
};
increment(1);
increment(2);
increment(3);
assert_eq!(counter.get(), 6);
// zero-overhead, probed: the flag-free wrapper is exactly T wide
assert_eq!(std::mem::size_of::<Cell<i32>>(), 4);
assert_eq!(std::mem::size_of::<Cell<u64>>(), 8);
// take() empties it (T::default goes in), get() confirms
let took = counter.take();
assert_eq!(took, 6);
assert_eq!(counter.get(), 0);
println!("Cell: shared mutation without borrows - the ch12 counter's whole trick");


**`Cell` is mutation without lending** — and the closure cell above is the whole story: `increment` captures `&counter` (a shared borrow), yet `set` mutates. At compile time the borrow checker saw only a shared reference; the mutation happened inside `Cell`'s API, which is allowed precisely because it only ever *copies*. The non-`Copy` wall: `Cell::new("s".to_string())` compiles, but `.get()` on it is E0507-flavoured refusal ("cannot move out of") — for owning types you want §6. The runtime cost is zero (`size_of` proved it); the design cost is that a `Cell` field makes the whole struct non-`Sync` (§9's thread boundary).


## 6. `RefCell`: The Runtime Borrow Check

`RefCell<T>` lends borrows **at runtime**, enforcing the same rules chapter 05 taught — many shared borrows *or* one exclusive borrow, never overlapping — but checking them per call and **panicking** on violation. `borrow()` hands you a `Ref<T>` (a guard deref-coercing to `&T`); `borrow_mut()` hands a `RefMut<T>`; each guard's `Drop` releases the loan (§3's trait doing the bookkeeping). Two `borrow()`s coexist; a `borrow_mut()` while any guard lives is the **already borrowed** panic, probed: the payload is a boxed **`String`** — `downcast_ref::<&str>()` returns `None`; `downcast_ref::<String>()` is the one that catches `"RefCell already borrowed"`. (Chapter 11's payload table gains a row: the std panics that build messages dynamically box `String`s.)

The size bill for the checking: `RefCell<i32>` is 16 bytes to `Cell<i32>`'s 4 — the borrow-flag state machine rides along. The design rule that follows: reach for `Cell` when `T: Copy` and the mutation is a field-sized swap; reach for `RefCell` when you must *lend* — iterators over a shared collection, a method that takes `&self` yet updates a cache, chapter 12's counter generalised beyond copies.

### The two loans, and the panic


In [ ]:
// 6.1 RefCell: two shared borrows fine; borrow_mut over one = runtime panic.
// first use in the notebook; later cells inherit it (a use line lives forever)
use std::cell::RefCell;
let book = RefCell::new(vec![10, 20]);
// shared + shared: the chapter-05 rule, honoured at runtime
{
    let a = book.borrow();
    let b = book.borrow();
    assert_eq!((a[0], b[1]), (10, 20));
} // both guards drop here (Drop again)
// exclusive through a shared binding:
book.borrow_mut().push(30); // the temporary guard drops at the semicolon
assert_eq!(book.borrow().len(), 3);
// the violation, caught the honest way:
let caught = std::panic::catch_unwind(std::panic::AssertUnwindSafe(|| {
    let _shared = book.borrow();
    let _excl = book.borrow_mut(); // panics: already borrowed
}));
let payload = caught.err().expect("double borrow must panic");
assert!(payload.downcast_ref::<&str>().is_none()); // NOT a literal &str
let msg = payload
    .downcast_ref::<String>()
    .map(|s| s.as_str())
    .unwrap_or("");
assert_eq!(msg, "RefCell already borrowed");
println!("RefCell: runtime borrow check is real - payload was a String");


Three reads. **`AssertUnwindSafe` is the catch's honesty tax**: a `&RefCell` shared into the closure crosses what the compiler considers un-unwind-safe state; asserting it is saying "I know the borrow flag survives an unwind" — true, and the guard `Drop`s still run. **The panic message is a `String`**, built at runtime by the borrow flag's state machine — which is why the `&str` downcast probe returned `None` where §6's prose promised it would (the chapter-11 lesson: *literal* panics box `&'static str`, *formatted/dynamic* ones box `String`). **The temporary-guard idiom** — `book.borrow_mut().push(..)` in one statement — is the everyday form: the guard lives exactly one statement, so the loan window is one line long and the next line's `borrow()` is uncontested.

## 7. `Rc<RefCell<T>>`: Sharing and Mutating

Stack them and you get the combination the borrow checker forbids and real programs keep needing: **many owners, and mutation through any of them**. `Rc` supplies the sharing; `RefCell` supplies the runtime-checked write access; the deref ladder (`&Rc<RefCell<T>>` → `&RefCell<T>` → `&T`) means `shared.borrow_mut()` just works. The canonical shape is a ledger several holders update through their own handles — no `&mut` in sight, the borrow rules still enforced, at runtime, per access.

The judgment §0 promised: this is *not* a garbage-collected object graph. `Rc<RefCell<T>>` is for **fan-in** — many owners of one thing — and it keeps every property ownership bought: deterministic drop (§3), no collection pauses, and panics instead of data races when the borrow rules break. The moment the structure needs *cycles* (a child pointing back at its parent), the strong counts deadlock the deallocation and §10's `Weak` enters. And the moment the sharing crosses a thread boundary, the whole stack is refused at compile time (§9).

### The ledger, three holders strong


In [ ]:
// 7.1 Rc<RefCell<T>>: many owners, mutation through any of them.
// (Rc and RefCell are already imported - a use line lives for the notebook.)
let ledger = Rc::new(RefCell::new(vec![100u32]));
assert_eq!(Rc::strong_count(&ledger), 1);
let clerk = Rc::clone(&ledger); // owner 2
let auditor = Rc::clone(&ledger); // owner 3
clerk.borrow_mut().push(250); // mutation through handle #2
auditor.borrow_mut().push(400); // and through #3
{
    let snap = ledger.borrow(); // owner 1 reads all three writes
    assert_eq!(snap[..], [100, 250, 400]);
}
assert_eq!(Rc::strong_count(&ledger), 3);
// the runtime borrow rule still applies ACROSS handles: this would panic -
// while one clerk holds borrow_mut, the auditor's borrow() is "already borrowed"
let excluded = std::panic::catch_unwind(std::panic::AssertUnwindSafe(|| {
    let _writing = clerk.borrow_mut();
    let _reading = auditor.borrow();
}));
assert!(excluded.is_err()); // same RefCell, two handles, one violation
println!("ledger: 3 owners, 3 writes, runtime rules enforced across handles");


The cell's last block is the insight that earns the section its own number: the borrow flag lives in the **`RefCell`**, not in any handle — so the violation is caught across *different owners' handles*, exactly as two threads sharing one mutex would serialise. Ownership, borrowing and panics compose predictably: three strong counts, one borrow flag, zero data races in single-threaded land.


## 8. `OnceCell` and `OnceLock`: Write Once

The last interior-mutation is the smallest: **write once, read forever**. `OnceCell::get_or_init(|| ..)` runs the closure only if the cell is still empty, stores the result, and returns `&T` — on every later call the closure is skipped and the stored value returns. The probed facts: two `get_or_init` calls with *different* closures both returned the first value ("computed", never "NOT-again"); a `set` on a filled cell returned `Err(the_value_you_tried_to_set)` — rejected, not panicking; `OnceLock` (the thread-safe sibling, §9's territory) reads back through `get() -> Option<&T>`. The everyday jobs: a lazily-built config table, a compiled regex cached on a struct, any "first caller pays, everyone after reads" pattern — without `Option<T>` plumbing or a `Mutex` in sight.

`OnceCell` is single-threaded (it lives in `std::cell`, next to `Cell` and `RefCell`); `OnceLock` is its `Sync` twin for `static`s and shared threads. Both are 8 bytes over their payload's handle cost, probed — the "once" state is one discriminant.

### The first writer wins


In [ ]:
// 8.1 OnceCell: init-on-first-use; the second init is never run.
use std::cell::OnceCell;
let config: OnceCell<String> = OnceCell::new();
let first = config.get_or_init(|| "computed-once".to_string());
let second = config.get_or_init(|| "NEVER-again".to_string());
assert_eq!(first, "computed-once");
assert_eq!(second, "computed-once"); // the same &T, closure skipped
// set() on a filled cell is a rejection, not a panic:
let refused = config.set("too-late".to_string());
assert_eq!(refused, Err("too-late".to_string()));
// the read side has no panic mode at all:
assert_eq!(config.get().map(|s| s.len()), Some(13));
println!("OnceCell: one init, one owner of the truth, refusals are Err values");


**`get_or_init` is the idiom; `set` is the assertion.** The cell above never checks `is_empty` by hand — the first call initialises, the second is a no-op read, and the only way to be wrong is to expect `set` to overwrite (it cannot; the value is `Err`-returned to you). Where does the *thread* line run? `OnceCell<T>` is `!Sync`, so a `static ONCE: OnceCell<..>` is refused at compile time — `OnceLock` is the spelling that compiles there (§9 shows the same split for `Rc` vs `Arc`).

## 9. `Arc` and the Thread Boundary

`Arc<T>` — atomically reference counted — is `Rc` with its counter bumped through atomic CPU instructions instead of plain arithmetic, which is exactly what lets handles on *different threads* agree on the count. The API surface is deliberately identical: `Arc::new`, `.clone()`, `strong_count`, deref coercion — a chapter-14 pointer through and through. The price of the atomics: slightly slower count updates, and the same read-only law as `Rc` (no `DerefMut`, no `&mut` through a shared handle).

The boundary itself is the compile-time wall worth seeing: `Rc<T>` is **neither `Send` nor `Sync`** (its counter is a plain, non-atomic integer — two threads incrementing it is a data race), so `Rc` inside a `thread::spawn` closure is E0277. The probed text for `Arc<RefCell<i32>>` is the chapter's bridge to chapter 16: "`RefCell<i32>` cannot be shared between threads safely" with the hint "**if you want to do aliasing and mutation between multiple threads, use `std::sync::RwLock` instead**" — the `Sync` shell (Arc) and the `Sync` mutation (RwLock/Mutex) each have to earn their place independently. Here the tour stops at the wall; chapter 16 owns everything past it.

### The wall, in one refusal


In [ ]:
// 9.1 Arc: the counted pointer that may cross threads - and the Rc that may not.
use std::sync::Arc;
use std::thread;
let shared = Arc::new(vec![1, 2, 3]);
let worker = Arc::clone(&shared);
let handle = thread::spawn(move || {
    // Arc<i32>-style reads cross the boundary; mutation would need Mutex/RwLock
    worker.iter().sum::<i32>()
});
let total = handle.join().expect("worker thread panicked");
assert_eq!(total, 6);
assert_eq!(Arc::strong_count(&shared), 1); // the worker's handle dropped at join
// The Rc refusal, spelled (a compile error, kept as the catalogue's text):
// let rc_shared = std::rc::Rc::new(0);
// thread::spawn(move || { *rc_shared; });
// error[E0277]: `Rc<i32>` cannot be sent between threads safely
println!("Arc crossed the thread boundary; Rc is refused there (E0277)");


Two reads on the boundary. **`Send` is about the *move into* the thread, `Sync` about sharing across them** — `Arc<T>: Send + Sync` (when `T` is), `Rc<T>: Send + Sync` never, `RefCell<T>: Send` but **not** `Sync` (two threads borrowing one flag is exactly the race the flag cannot arbitrate). **The E0277 note names the cure**, and it is chapter 16's whole first act: `RwLock`/`Mutex` for shared mutation, `Arc` for the shared *ownership*. A `spawn`ed closure also requires `'static` — the thread may outlive the caller's stack — which is why the heap-owned `Arc` (not a borrowed slice) is what gets moved in.


## 10. Cycles and `Weak`: The Leak

`Rc` frees memory by counting: each handle's `Drop` decrements, the payload frees at zero. A **reference cycle** — A owns B, B owns A — freezes the arithmetic: both counts rest at 1 forever, neither `Drop` ever runs, and the memory leaks *deterministically* (no collector will ever come). The probe made this visible with `Drop`-printing nodes: a **broken chain** (B → A, no back-edge) dropped both nodes the moment the handles fell out of scope; the **cycle** (A → B → A, both strong counts 2) printed nothing — the destructors never ran.

`Weak<T>` is the cycle-breaker: `Rc::downgrade(&r)` produces a *non-owning* watcher — it bumps **weak_count**, not strong_count, and holding it keeps no one alive. Reading through a `Weak` requires **`upgrade()`**, which returns `Option<Rc<T>>`: `Some` while a strong owner exists, `None` the instant the last one drops (probed: upgrade worked while `strong` lived, returned `None` after it dropped). The idiom: **ownership flows one way, `Weak` flows back** — child-to-parent edges, caches, observer lists.

### The leak, the break, the upgrade dance


In [ ]:
// 10.1 Weak: a non-owning back-edge breaks the cycle; upgrade reads if alive.
// (Rc and RefCell are already imported; Weak joins them for this cell.)
use std::rc::Weak;

struct Node {
    name: &'static str,
    parent: RefCell<Weak<Node>>, // the back-edge: watched, not owned
    children: RefCell<Vec<Rc<Node>>>, // ownership flows down only
}
impl Node {
    fn new(name: &'static str) -> Rc<Node> {
        Rc::new(Node {
            name,
            parent: RefCell::new(Weak::new()),
            children: RefCell::new(vec![]),
        })
    }
}
impl Drop for Node {
    fn drop(&mut self) {
        println!("[drop] Node {}", self.name);
    }
}
let upgrade_works = {
    let root = Node::new("root");
    let leaf = Node::new("leaf");
    *leaf.parent.borrow_mut() = Rc::downgrade(&root); // Weak: no strong bump
    root.children.borrow_mut().push(Rc::clone(&leaf));
    assert_eq!(Rc::strong_count(&root), 1); // the back-edge did not count
    assert_eq!(Rc::weak_count(&root), 1);
    // reading through the Weak while the parent lives:
    let seen = leaf.parent.borrow().upgrade();
    let name = seen.as_ref().map(|r| r.name);
    name
}; // brace: root, then leaf drop - BOTH [drop] lines print, so no leak
assert_eq!(upgrade_works, Some("root"));
println!("(scope ended: [drop] Node leaf, [drop] Node root printed above)");


When the cell's bindings are discarded, both `[drop]` lines print — **root and leaf free cleanly** because the only back-edge is a `Weak`. Contrast the strong cycle: swap `Weak<Node>` for `Rc<Node>` in `parent`, and the same cell leaks silently — both nodes' destructors never run, as the probe showed. Three rules carry the design: **`downgrade` never keeps anyone alive** (it is the receipt, not the ownership); **`upgrade` is the only door** (an `Option` per read — the price of not leaking); and **a `Weak` does not deref** — there is no coercion through it, because the value it watches may already be gone. The probed `Rc::weak_count` rows: a fresh `Rc` reports 0; one `downgrade` reports 1 — even as strong stays 1.

## 11. Reading the Compiler: E-Codes

The chapter's diagnostics are the pointer contracts talking: an indirection demanded, an orphan rule enforced, a borrow refused at *runtime* instead. Every wording below was captured from this exact toolchain.

| Diagnostic | Trigger | Message (captured) | Fix |
|---|---|---|---|
| **E0072** | a type containing itself (`Cons(i32, Menu)`) | ``recursive type `Menu` has infinite size`` + ``insert some indirection (e.g., a `Box`, `Rc`, or `&`)`` | `Box<Menu>` — §1.1's list |
| **E0046** | `impl Deref` without `type Target` | ``not all trait items implemented, missing: `Target` `` | add the associated type (§2.1) — not an E0106 lifetime matter |
| **E0117 + E0120** | `impl Drop for i32` | ``only traits defined in the current crate can be implemented for primitive types`` + ``the `Drop` trait may only be implemented for local structs, enums, and unions`` | wrap the foreign type in a newtype and drop that (§3) |
| **E0119** | a second `impl Drop for Meter` | ``conflicting implementations of trait `Drop` for type `Meter` `` | one destructor per type, ever (§3) |
| **E0597** | a coerced `&str` outliving its `Box` | ``` `b` does not live long enough ``` + `` `b` dropped here while still borrowed `` | return an owned `String`, or let the `Box` outlive the borrow (§2) |
| **E0107** | `Box` without its type argument | ``missing generics for struct `Box`` `` | spell `Box<T>` — the pointer's target is part of its type |
| **E0277** | `Rc`/`RefCell` moved into `thread::spawn` | `` `RefCell<i32>` cannot be shared between threads safely`` + ``use `std::sync::RwLock` instead`` | `Arc` for ownership, `Mutex`/`RwLock` for mutation — chapter 16 (§9) |

Three worth dwelling on. **E0072 is the recursion wall**: size must be computable at compile time, self-containment is not, and the fix the diagnostic proposes is the smart-pointer chapter in one clause — *indirection moves the tail to the heap*. **E0117+E0120 is chapter 10's orphan rule with `Drop` sharpening it**: even a *local* trait impl cannot attach destructors to primitives, because `drop` runs implicitly on every `i32` in every program — the pair arrives together, and the newtype is the sanctioned door. **The E0277 row is a fork in the track**: the same diagnostic that refused your non-`Sync` state in `spawn` now tells you which tool unlocks it — and `RefCell`'s own failure mode stays single-threaded by design, panicking instead of racing.

### The catalogue, exercised


In [ ]:
// 11.1 The pointer diagnostics, dodged in one cell each.
// E0072 dodged: the recursive tail lives behind a Box (size = handle size).
enum Chain {
    Link(u32, Box<Chain>),
    End,
}
let chain = Chain::Link(1, Box::new(Chain::Link(2, Box::new(Chain::End))));
fn chain_sum(chain: &Chain) -> u32 {
    match chain {
        Chain::Link(n, next) => n + chain_sum(next),
        Chain::End => 0,
    }
}
assert_eq!(chain_sum(&chain), 3); // the fields are read, not just built
// E0046 dodged: Target spelled, the coercion machine satisfied.
// (Deref and DerefMut were imported in section 2.)
struct Wrap {
    inner: String,
}
impl Deref for Wrap {
    type Target = String;
    fn deref(&self) -> &String {
        &self.inner
    }
}
let w = Wrap { inner: "ok".to_string() };
assert_eq!(w.len(), 2); // deref coercion through our own Target
// E0107 dodged: the type argument is part of the pointer's type.
let sized: Box<u32> = Box::new(9);
assert_eq!(*sized, 9);
// E0117/E0120 dodged: Drop only on our own local type (Gauge, section 3).
// E0597 dodged: the borrow never outlives the owner.
let boxed = Box::new("short".to_string());
let coerced: &str = &boxed; // lives exactly as long as `boxed` does
assert_eq!(coerced, "short");
// E0277 dodged: single-threaded sharing stays in single-threaded land.
let local = std::rc::Rc::new(1);
assert_eq!(*local, 1);
println!("seven diagnostics known: E0072, E0046, E0117/E0120, E0119, E0597, E0107, E0277");


The dodge cell's quiet star is the **E0597 row**: `&boxed` coerces to `&str` and *stays* inside `boxed`'s scope — the borrow checker's lifetime rule, chapter 05's oldest lesson, now protecting a heap payload. The runtime half of the catalogue — `RefCell`'s **already borrowed** panic and `OnceCell`'s `Err`-returning `set` — cannot appear in a compile-error table because neither is one; their refusals are §6's and §8's subject, and the practice twin's checks exercise them the only way possible: `catch_unwind`, downcast, assert the message.


## 12. Mental Model: The Receipts

| Question | Answer with | Where |
|---|---|---|
| "How do I put one value on the heap?" | `Box::new(v)` — sole owner, pointer-sized handle | §1 |
| "How do I make a recursive type legal?" | indirection: `Box` in the recursive tail (E0072 says so) | §1, §11 |
| "Why do smart pointers read like plain values?" | `Deref`/`DerefMut` — the compiler walks the coercion ladder | §2 |
| "When does cleanup run?" | `Drop` — LIFO, deterministic; `mem::drop` for early | §3 |
| "How do many owners share one value?" | `Rc::clone` bumps the strong count; payload frees at zero | §4 |
| "How do I mutate through a shared reference?" | `Cell` (copy in/out) or `RefCell` (runtime-checked loans) | §5, §6 |
| "Which runtime rule broke?" | the borrow flag: `already borrowed` panic, payload a `String` | §6 |
| "How do I share *and* mutate?" | `Rc<RefCell<T>>` — one borrow flag, many handles | §7 |
| "How do I write once, read forever?" | `OnceCell::get_or_init` — second init never runs | §8 |
| "What crosses threads?" | `Arc` (ownership) + `Mutex`/`RwLock` (mutation) — never `Rc`/`Cell` | §9 |
| "How do I point back without owning?" | `Weak` + `upgrade() -> Option` — the cycle-breaker | §10 |

Read the table as the receipts ledger: every relaxation of chapter 05's rules comes with a paper trail — `Box` pays with indirection, `Rc` with a count and read-only access, `RefCell` with runtime panics, `Arc` with atomic instructions, `Weak` with an `Option` per read. Python's GC hides all receipts and pays in pauses; C++'s `shared_ptr` has the count but no borrow flag on the way in; Rust makes each cost a *type you chose*, visible at every callsite — which is why the leak in §10 is a graph-shape bug, not a runtime mystery.

## 13. Cheat Sheet

| Task | Code |
|---|---|
| sole heap owner | `let b = Box::new(v);` |
| fix the recursion wall | `enum List { Cons(i32, Box<List>), Nil }` |
| move the payload out | `let v: T = *b;` — `Box` only, owners only |
| read as the target | `impl Deref for W { type Target = T; fn deref(&self) -> &T }` |
| mutate through the wrapper | `impl DerefMut` — then `w.method()` coerces `&mut` too |
| run cleanup now | `std::mem::drop(v);` — the destructor is immediate |
| the destructor | `impl Drop for T { fn drop(&mut self) { .. } }` — LIFO |
| share one value | `Rc::new(v)`, then `Rc::clone(&r)` per owner |
| count the owners | `Rc::strong_count(&r)` / `Rc::weak_count(&r)` |
| same allocation? | `Rc::ptr_eq(&a, &b)` |
| steal the last handle | `Rc::try_unwrap(r).ok()` — `Err` while shared |
| shared mutation, copies | `Cell::new(0)` → `.get()` / `.set(v)` / `.take()` |
| shared mutation, loans | `RefCell::new(v)` → `.borrow()` / `.borrow_mut()` |
| the guard window | one statement: `cell.borrow_mut().push(x);` |
| the violation | two conflicting guards → panic `"RefCell already borrowed"` (String payload) |
| share + mutate | `Rc<RefCell<Vec<T>>>` — `handle.borrow_mut()` through any clone |
| write once | `OnceCell::get_or_init(\|\| build())` — later calls skip the closure |
| overwrite refusal | `set(v)` on filled → `Err(v)` |
| cross a thread | `Arc::new(v)` + `Arc::clone` into `thread::spawn` |
| point back safely | `parent: RefCell<Weak<Node>>` + `upgrade().map(..)` |
| break a cycle | ownership down, `Weak` back — strong counts must form a DAG |

## 14. Review

Answer from memory; each one is a §-numbered cell away.

1. `Box<[u8; 100]>` and `Box<i32>` are both 8 bytes. What does the handle hold, where does the payload live, and which probed size shows the *fat* exception? (§1)
2. `enum Menu { Course(i32, Menu) }` is refused. Which E-code fires, what exactly does the diagnostic propose, and why does `Box` satisfy it? (§1, §11)
3. `takes_str(&m)` accepted a `&MeterBox`. Which trait item made the coercion legal, which second trait let `m.push_str(..)` compile, and what is the multi-hop form of the ladder? (§2)
4. `mem::drop(inner)` printed its `[drop]` line before the next `println!`. What does that prove about `Drop` timing, and which order do `a`, `b`, `c` drop in at a scope's end? (§3)
5. `impl Drop for i32` drew *two* E-codes. Name both and the rule each encodes; why is even a local trait impl forbidden from touching primitives? (§3, §11)
6. `Rc::clone` doubled the strong count without copying the payload. Which function proved the two handles are the same allocation, and which two functions refused mutation while it was shared? (§4)
7. The ch12 laziness counter mutated through a shared borrow. Which type made it legal, why does it need `T: Copy`, and what did `size_of::<Cell<i32>>()` prove about its cost? (§5)
8. Two `borrow()` calls coexisted; `borrow_mut` over one panicked. What exactly does the borrow flag enforce, what is the panic's payload *type*, and why did chapter 11's `&str` downcast habit miss it? (§6)
9. `clerk.borrow_mut()` and `auditor.borrow()` on two `Rc` clones of one `RefCell` panicked. Where does the borrow flag live, and what does that imply about enforcing rules across handles? (§7)
10. The second `get_or_init` never ran its closure, and `set` returned `Err(v)`. Which guarantees does `OnceCell` give, and which spelling compiles in a `static` — and why? (§8, §9)
11. `Arc<RefCell<i32>>` into `thread::spawn` was E0277. What is `RefCell` missing, which type does the diagnostic suggest for mutation, and what is `Arc` itself for? (§9, §11)
12. The cycle's two nodes never dropped; the `Weak` back-edge's did. Why do strong counts freeze a cycle, which method is the only door through a `Weak`, and what does its `Option` return mean? (§10)

### What Comes Next

The receipts ledger is complete; what follows points it at bigger machinery:

| Chapter | Title | What it settles |
|---|---|---|
| **15** | **Data Types and Their Libraries** | `chrono`/`time`, `polars`, `uuid`, `rust_decimal` — the crate as the unit you *consume* |
| 16 | Concurrency and Async/Await | the wall from §9 comes down: `Send`/`Sync` in full, `Mutex`, channels, `tokio` |
| 17 | Capstone: Applying the Track | every receipt cashed at once — a real project wired end to end |

**Carry forward.** You should now be able to do these without looking anything up:

- choose the pointer by its contract: `Box` to own and indirect, `Rc` to fan ownership in, `Arc` to cross threads
- implement `Deref`/`DerefMut` for a newtype and explain every coercion the compiler then performs
- place `Drop` on a local type and predict the drop order of any scope — including early `mem::drop`
- pick `Cell` for copy-sized mutation and `RefCell` for loans, and say which borrow rule panics at runtime
- judge when `Rc<RefCell<T>>` is the answer and when it is the shape that needs `Weak` — or `Mutex`
- break a reference cycle with a `Weak` back-edge and prove the fix by the `[drop]` lines
- read the seven E-codes of this chapter and name the relaxation each one is defending
